# Análisis No Supervisado de Fraude — Ecommerce
### Detección de patrones sin necesidad de etiqueta
**Scotiabank Peru — Prevención de Fraude**

---

## El problema

Las transacciones marcadas como **N (Normal)** no son necesariamente legítimas — son transacciones **aún no revisadas**.
El fraude más peligroso vive en la N: llegó, pasó las reglas actuales, y nadie lo detectó todavía.

Este notebook implementa **dos tracks complementarios**:

| Track | Enfoque | Requiere etiqueta F |
|---|---|---|
| **A — Supervisado** | Confirmar qué ya sabemos detectar (F vs N) | ✅ Sí |
| **B — No supervisado** | Detectar patrones sospechosos en las N | ❌ No |

Al final genera una **lista priorizada de transacciones N** para revisión proactiva del analista,
antes de que lleguen los reclamos.

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', font_scale=0.9)

BASE_DIR = Path('..').resolve()
sys.path.insert(0, str(BASE_DIR / 'scripts'))

from config import COLS, PARQUET_FEATURES, COMERCIO_NOMBRE
print(f'Comercio: {COMERCIO_NOMBRE}')
print(f'Base dir: {BASE_DIR}')

## 1. Carga de datos

Se intenta primero el parquet con ML (`consolidado_features_ml.parquet`) que incluye
`ANOMALY_SCORE` y `CLUSTER_HDBSCAN`. Si no existe, se usa el parquet estándar.

In [ ]:
parquet_ml  = BASE_DIR / 'data' / 'consolidado_features_ml.parquet'
parquet_std = PARQUET_FEATURES

if parquet_ml.exists():
    df = pd.read_parquet(parquet_ml)
    tiene_ml = True
    print(f'✅ Cargado parquet ML: {parquet_ml.name}')
else:
    df = pd.read_parquet(parquet_std)
    tiene_ml = False
    print(f'⚠️  Parquet estándar (sin ANOMALY_SCORE)')
    print(f'   Para activar ML: python ml/clustering_fraude.py')

C         = COLS
col_ind   = C['indicador']
col_monto = C['monto']
col_cli   = C['id_cliente']
col_bin   = C.get('bin', '')

df[col_monto] = pd.to_numeric(df[col_monto], errors='coerce')

n_fraudes    = int((df[col_ind] == 'F').sum())
tasa_global  = round(n_fraudes / len(df) * 100, 2)

print(f'\nFilas             : {len(df):,}')
print(f'Columnas          : {df.shape[1]}')
print(f'Monto total  (S/) : {df[col_monto].sum():,.2f}')
print(f'Fraudes (F)       : {n_fraudes:,}')
print(f'Tasa global fraude: {tasa_global}%')

## 2. Distribución de indicadores — el tamaño del problema

In [ ]:
IND_DESC = {'F': 'Fraude confirmado', 'G': 'Buena (liberada)',
            'N': 'Normal (no revisada)', 'D': 'Descarte', 'P': 'Pendiente'}
IND_COL  = {'F': '#C00000', 'G': '#70AD47', 'N': '#BDD7EE', 'D': '#FFD966', 'P': '#FF8C00'}

ind_counts = df[col_ind].value_counts()
ind_pct    = (ind_counts / len(df) * 100).round(2)

resumen_ind = pd.DataFrame({
    'Indicador'   : ind_counts.index,
    'Descripción' : ind_counts.index.map(IND_DESC).fillna('Otro'),
    'N'           : ind_counts.values,
    'Pct%'        : ind_pct.values,
})
print(resumen_ind.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(ind_counts.index,
              ind_counts.values,
              color=[IND_COL.get(i, '#AAAAAA') for i in ind_counts.index],
              edgecolor='white', linewidth=0.5)
ax.bar_label(bars, labels=[f'{p:.1f}%' for p in ind_pct.values], padding=4, fontsize=9)
ax.set_title(f'Distribución de indicadores — {COMERCIO_NOMBRE}', fontweight='bold')
ax.set_xlabel('Indicador')
ax.set_ylabel('Transacciones')
plt.tight_layout()
plt.show()

n_N = int((df[col_ind] == 'N').sum())
print(f'\n⚠️  Hay {n_N:,} transacciones N. Si solo el {tasa_global}% son fraude,')
print(f'   podrían existir ~{int(n_N * tasa_global / 100):,} fraudes sin detectar dentro de las N.')

## 3. Variables disponibles por bloque

El parquet tiene variables organizadas en bloques temáticos.
Isolation Forest usará las numéricas de todos los bloques.

In [ ]:
BLOQUES = {
    'B — Temporal'            : ['HORA_DIA','DIA_SEMANA','ES_FIN_SEMANA','ES_MADRUGADA','FRANJA_HORARIA'],
    'D+E — Velocidad cliente' : ['TRX_CLIENTE_5MIN','TRX_CLIENTE_1H','GAP_MINUTOS','ACELERACION_MONTO'],
    'F+N — Perfil cliente'    : ['FLAG_REINCIDENTE','ZSCORE_MONTO_CLI_COMERCIO','N_FRAUDES_CLIENTE_PERIODO'],
    'G — Perfil comercio/MCC' : ['TASA_FRAUDE_MCC','RANKING_COMERCIO','FLAG_PAIS_INUSUAL'],
    'H — Señales monto'       : ['ZSCORE_MONTO_COMERCIO','DECIL_MONTO','FLAG_MONTO_REDONDO'],
    'I — Card testing'        : ['FLAG_BIN10_REPETIDO_DIA','FLAG_BIN11_REPETIDO_DIA',
                                  'FLAG_BIN12_REPETIDO_DIA','FLAG_VEN_CONCENTRADA_BIN'],
    'K+L — Flags/Score'       : ['SCORE_RIESGO','PERFIL_RIESGO','FLAG_RAFAGA_5MIN'],
    'M — Score Monitor'       : ['SCORE_MON_NORM','FLAG_SCORE_RIESGO_MON_ALTO'],
    'Q — Velocidad BIN'       : ['TRX_BIN_1H','MNT_BIN_24H','CLIENTES_BIN_DIA',
                                  'FLAG_RAFAGA_BIN_1H','FLAG_CLIENTES_BIN_ALTO'],
    'R — Generación robótica' : ['CV_MONTO_BIN_DIA','N_TARJETAS_MISMO_MONTO_BIN',
                                  'FLAG_MONTO_ROBOTICO_BIN'],
    'P — ML (IF+HDBSCAN)'     : ['ANOMALY_SCORE','FLAG_ANOMALIA_IF','CLUSTER_HDBSCAN'],
}

print(f'{'Bloque':<30} {'Disponibles':>11} {'Total':>7}')
print('─' * 52)
for bloque, vars_bloque in BLOQUES.items():
    disponibles = sum(1 for v in vars_bloque if v in df.columns)
    total       = len(vars_bloque)
    estado      = '✅' if disponibles == total else f'⚠️  {disponibles}/{total}'
    print(f'{bloque:<30} {estado:>11} {total:>7}')

---
## TRACK A — Lo que ya sabemos detectar (con etiqueta)

Comparar F vs N en las variables más discriminativas.
Si F tiene valores más altos que N → la variable es útil para una regla.

In [ ]:
VARS_TRACK_A = [
    ('TRX_CLIENTE_5MIN',     'Txn en 5 min'),
    ('TRX_CLIENTE_1H',       'Txn en 1h'),
    ('GAP_MINUTOS',          'GAP entre txn (min)'),
    ('SCORE_RIESGO',         'Score de riesgo 0-11'),
    ('ZSCORE_MONTO_CLIENTE', 'Z-score monto cliente'),
]
vars_ok = [(v, lbl) for v, lbl in VARS_TRACK_A if v in df.columns]

fig, axes = plt.subplots(1, len(vars_ok), figsize=(4 * len(vars_ok), 4))
if len(vars_ok) == 1:
    axes = [axes]

for ax, (var, label) in zip(axes, vars_ok):
    for ind, color, alpha in [('F', '#C00000', 0.7), ('N', '#2E75B6', 0.4)]:
        if ind in df[col_ind].unique():
            _s = df.loc[df[col_ind] == ind, var].dropna()
            _s = _s[_s <= _s.quantile(0.97)]
            ax.hist(_s, bins=30, alpha=alpha, color=color, label=ind, density=True)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Track A — Distribución F (rojo) vs N (azul) en variables clave',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Tabla comparativa
print('\nComparativa F vs N (media y mediana):')
rows = []
for var, label in vars_ok:
    for ind in ['F', 'N']:
        _s = df.loc[df[col_ind] == ind, var].dropna()
        rows.append({'Variable': label, 'Indicador': ind,
                     'Media': round(_s.mean(), 2), 'Mediana': round(_s.median(), 2),
                     'P90': round(_s.quantile(0.9), 2)})
print(pd.DataFrame(rows).to_string(index=False))

---
## TRACK B — Detección sin etiqueta

### B.1 — Isolation Forest

Isolation Forest puntúa cada transacción individualmente según qué tan difícil es **aislarla** del resto.
Una txn que se separa fácilmente del grupo = anómala = sospechosa.

- `ANOMALY_SCORE` cercano a **1** = muy anómala
- `ANOMALY_SCORE` cercano a **0** = comportamiento típico

**No usa la etiqueta F para nada** — trabaja sobre todas las transacciones incluyendo las N.

In [ ]:
if tiene_ml and 'ANOMALY_SCORE' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Histograma superpuesto F/G/N
    ax = axes[0]
    for ind, color, alpha in [('N','#BDD7EE',0.6), ('G','#70AD47',0.6), ('F','#C00000',0.8)]:
        if ind in df[col_ind].unique():
            _s = df.loc[df[col_ind] == ind, 'ANOMALY_SCORE'].dropna()
            ax.hist(_s, bins=40, alpha=alpha, color=color, label=ind, density=True)
    ax.set_title('ANOMALY_SCORE por indicador\n(si F está más a la derecha = IF discrimina bien)',
                fontweight='bold')
    ax.set_xlabel('ANOMALY_SCORE (mayor = más anómalo)')
    ax.legend()

    # Boxplot
    ax2 = axes[1]
    _grps  = [df.loc[df[col_ind]==i,'ANOMALY_SCORE'].dropna().values
               for i in ['F','G','N'] if i in df[col_ind].unique()]
    _labs  = [i for i in ['F','G','N'] if i in df[col_ind].unique()]
    bp = ax2.boxplot(_grps, labels=_labs, patch_artist=True, notch=True)
    for patch, color in zip(bp['boxes'], ['#C00000','#70AD47','#BDD7EE'][:len(_grps)]):
        patch.set_facecolor(color); patch.set_alpha(0.6)
    ax2.set_title('Dispersión ANOMALY_SCORE', fontweight='bold')
    ax2.set_ylabel('ANOMALY_SCORE')

    plt.tight_layout()
    plt.show()

    # Tabla resumen
    rows_if = []
    for ind in ['F','G','N','D','P']:
        _sub = df[df[col_ind] == ind]
        _s   = _sub['ANOMALY_SCORE'].dropna()
        if len(_s) == 0: continue
        rows_if.append({
            'Indicador'    : ind,
            'N_txn'        : len(_s),
            'Score_medio'  : round(_s.mean(), 4),
            'Score_mediana': round(_s.median(), 4),
            'Score_P90'    : round(_s.quantile(0.9), 4),
            'N_anomalias'  : int((_sub['FLAG_ANOMALIA_IF'] == 1).sum()),
            'Pct_anomalias%': round((_sub['FLAG_ANOMALIA_IF'] == 1).mean() * 100, 1),
        })
    print('\nResumen ANOMALY_SCORE por indicador:')
    print(pd.DataFrame(rows_if).to_string(index=False))
else:
    print('ANOMALY_SCORE no disponible.')
    print('Ejecuta: python ml/clustering_fraude.py')

In [ ]:
# Top 20 transacciones N con mayor ANOMALY_SCORE
if tiene_ml and 'ANOMALY_SCORE' in df.columns:
    _cols_top = [c for c in [
        col_ind, col_bin, col_monto,
        'ANOMALY_SCORE', 'SCORE_RIESGO', 'PERFIL_RIESGO',
        'TRX_CLIENTE_5MIN', 'TRX_BIN_1H', 'CLIENTES_BIN_DIA',
        'FLAG_MONTO_ROBOTICO_BIN', 'FLAG_BIN11_REPETIDO_DIA',
        'MARCA_TARJETA', 'TIPO_PRODUCTO_TEXTO',
    ] if c in df.columns]

    top_n = (df[df[col_ind] == 'N']
             .nlargest(20, 'ANOMALY_SCORE')[_cols_top]
             .reset_index(drop=True))
    top_n.index += 1
    print('TOP 20 transacciones N más sospechosas según Isolation Forest:')
    print(top_n.to_string())

### B.2 — HDBSCAN — Grupos de ataque

HDBSCAN agrupa transacciones similares. Un cluster con alta tasa de fraude = patrón reconocible.
El cluster **-1** es ruido: transacciones que no encajan en ningún grupo (pueden ser fraudes muy específicos).

In [ ]:
if tiene_ml and 'CLUSTER_HDBSCAN' in df.columns:
    rows_cl = []
    for cl_id in sorted(df['CLUSTER_HDBSCAN'].unique()):
        _sub = df[df['CLUSTER_HDBSCAN'] == cl_id]
        _row = {
            'Cluster'   : int(cl_id),
            'Etiqueta'  : 'RUIDO/OUTLIER' if cl_id == -1 else f'Cluster {cl_id}',
            'N_txn'     : len(_sub),
            'Pct%'      : round(len(_sub) / len(df) * 100, 1),
            'Monto_prom': round(_sub[col_monto].mean(), 2),
            'N_Fraude'  : int((_sub[col_ind] == 'F').sum()),
            'TASA_F%'   : round((_sub[col_ind] == 'F').mean() * 100, 2),
        }
        if 'SCORE_RIESGO' in _sub.columns:
            _row['Score_prom'] = round(_sub['SCORE_RIESGO'].mean(), 2)
        rows_cl.append(_row)

    df_cl = pd.DataFrame(rows_cl)
    print(df_cl.to_string(index=False))

    # Gráfico volumen vs tasa de fraude por cluster
    fig, ax = plt.subplots(figsize=(10, 4))
    ax2 = ax.twinx()
    _x = [str(r['Cluster']) for r in rows_cl]
    ax.bar(_x,  [r['N_txn']   for r in rows_cl], color='#2E75B6', alpha=0.5, label='N txn')
    ax2.plot(_x, [r['TASA_F%'] for r in rows_cl], 'o-', color='#C00000',
             linewidth=2, markersize=6, label='Tasa F%')
    ax.set_title('Clusters HDBSCAN — volumen vs tasa de fraude', fontweight='bold')
    ax.set_xlabel('Cluster (-1 = ruido)'); ax.set_ylabel('N transacciones')
    ax2.set_ylabel('Tasa fraude %'); ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
else:
    print('CLUSTER_HDBSCAN no disponible.')
    print('Ejecuta: python ml/clustering_fraude.py')

---
## TRACK B.3 — Patrones de BIN sin etiqueta

Las variables del Bloque Q y R detectan ataques sin necesitar la etiqueta F.
Funcionan sobre **todas las transacciones incluyendo las N**.

In [ ]:
# Velocidad por BIN — Top BINs con mayor actividad por día
if 'TRX_BIN_1H' in df.columns and col_bin and col_bin in df.columns:

    top_bins = (
        df.groupby(col_bin).agg(
            TRX_BIN_1H_max       = ('TRX_BIN_1H',      'max'),
            CLIENTES_BIN_DIA_max = ('CLIENTES_BIN_DIA', 'max'),
            MNT_BIN_24H_max      = ('MNT_BIN_24H',      'max'),
            N_total              = (col_monto,           'count'),
            N_fraude             = ('ES_FRAUDE',         'sum'),
        )
        .assign(TASA_F=lambda x: (x['N_fraude'] / x['N_total'] * 100).round(2))
        .sort_values('CLIENTES_BIN_DIA_max', ascending=False)
        .head(15).reset_index()
    )

    # ── Tabla resumen ─────────────────────────────────────────────────────────
    print('=' * 75)
    print('TOP 15 BINs POR CLIENTES DISTINTOS EN UN DÍA (señal de ataque distribuido)')
    print('=' * 75)
    print(f'{"BIN":<15} {"Max_Cli/Día":>11} {"Max_TRX_1h":>11} {"Max_MNT_24h":>12} {"N_total":>8} {"N_F":>6} {"TASA_F%":>8}')
    print('-' * 75)
    for _, r in top_bins.iterrows():
        print(f'{str(r[col_bin]):<15} {int(r["CLIENTES_BIN_DIA_max"]):>11,} {int(r["TRX_BIN_1H_max"]):>11,} '
              f'{r["MNT_BIN_24H_max"]:>12,.0f} {int(r["N_total"]):>8,} {int(r["N_fraude"]):>6,} {r["TASA_F"]:>8.2f}%')
    print('=' * 75)
    print()
    print('► Columna clave: Max_Cli/Día — si un BIN tiene 40+ clientes en un solo día')
    print('  cuando su patrón normal es 1-5, es un ataque distribuido (no bot, sino')
    print('  tarjetas generadas del mismo BIN usadas en distintas horas del día).')

    # ── Heatmap BIN × Día ─────────────────────────────────────────────────────
    if 'FECHA_DIA' in df.columns:
        top10_bins = top_bins[col_bin].head(10).tolist()
        pivot = (
            df[df[col_bin].isin(top10_bins)]
            .pivot_table(index=col_bin, columns='FECHA_DIA',
                         values=col_monto, aggfunc='count', fill_value=0)
        )

        if len(pivot) > 0:
            n_bins  = len(pivot)
            n_dias  = len(pivot.columns)
            alto    = max(5, n_bins * 0.9)
            ancho   = max(14, n_dias * 0.55)

            fig, ax = plt.subplots(figsize=(ancho, alto))
            sns.heatmap(
                pivot, cmap='YlOrRd', ax=ax,
                linewidths=0.4, linecolor='#cccccc',
                cbar_kws={'label': 'N transacciones', 'shrink': 0.6},
                annot=(n_bins <= 10 and n_dias <= 30),   # números dentro si es chico
                fmt='d', annot_kws={'size': 7},
            )

            ax.set_title(
                'Heatmap BIN × Día  —  Top 10 BINs por clientes distintos en un día
'
                'Cuadros rojos = volumen atípico = posible ataque distribuido',
                fontweight='bold', fontsize=12, pad=14
            )
            ax.set_xlabel('Fecha', fontsize=10)
            ax.set_ylabel('BIN', fontsize=10)

            # BINs en eje Y — horizontales y legibles
            ax.set_yticklabels(
                ax.get_yticklabels(),
                rotation=0, fontsize=9, fontfamily='monospace'
            )
            # Fechas en eje X — rotadas 45° para no solaparse
            ax.set_xticklabels(
                [t.get_text()[:10] for t in ax.get_xticklabels()],
                rotation=45, ha='right', fontsize=8
            )

            plt.tight_layout()
            plt.show()

            # ── Mini tabla: días con spike para cada BIN ──────────────────────
            print()
            print('DÍAS CON VOLUMEN ATÍPICO (> 3× la mediana del BIN):')
            print('-' * 60)
            for bin_val in top10_bins:
                if bin_val not in pivot.index: continue
                serie = pivot.loc[bin_val]
                mediana = serie[serie > 0].median() if (serie > 0).sum() > 1 else 1
                dias_spike = serie[serie > mediana * 3].sort_values(ascending=False)
                if len(dias_spike) > 0:
                    dias_str = '  |  '.join(
                        [f"{str(d)[:10]}: {v} txn" for d, v in dias_spike.head(5).items()]
                    )
                    print(f'  {str(bin_val):<15}  →  {dias_str}')
            print('-' * 60)
else:
    print('TRX_BIN_1H no disponible — ejecuta feature_engineering.py')


In [ ]:
# Generación robótica — monto idéntico entre tarjetas del mismo BIN
if 'FLAG_MONTO_ROBOTICO_BIN' in df.columns:
    n_rob = int(df['FLAG_MONTO_ROBOTICO_BIN'].sum())
    print(f'Transacciones con patrón robótico: {n_rob:,}')

    if n_rob > 0:
        tasa_rob  = df.loc[df['FLAG_MONTO_ROBOTICO_BIN']==1, 'ES_FRAUDE'].mean() * 100
        lift_rob  = round(tasa_rob / tasa_global, 1) if tasa_global > 0 else 0
        print(f'Tasa fraude en robóticas  : {tasa_rob:.1f}%')
        print(f'Tasa fraude global        : {tasa_global:.1f}%')
        print(f'LIFT                      : {lift_rob}x  ← si > 3 = señal válida para regla')

        rob_det = (
            df[df['FLAG_MONTO_ROBOTICO_BIN'] == 1]
            .groupby(col_bin).agg(
                N_txn      = (col_monto,                  'count'),
                Montos_dist= (col_monto,                  'nunique'),
                CV_monto   = ('CV_MONTO_BIN_DIA',         'mean'),
                N_tarjetas = ('N_TARJETAS_MISMO_MONTO_BIN','max'),
                N_fraude   = ('ES_FRAUDE',                'sum'),
            )
            .assign(TASA_F=lambda x: (x['N_fraude']/x['N_txn']*100).round(1))
            .sort_values('N_txn', ascending=False).head(10).reset_index()
        )
        print('\nTop 10 BINs con patrón robótico:')
        print(rob_det.to_string(index=False))

    # Distribución CV
    if 'CV_MONTO_BIN_DIA' in df.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        _cv = df['CV_MONTO_BIN_DIA'].dropna()
        _cv = _cv[_cv <= _cv.quantile(0.99)]
        ax.hist(_cv, bins=50, color='#2E75B6', alpha=0.7, edgecolor='white')
        ax.axvline(0.05, color='#C00000', linestyle='--', linewidth=2,
                   label='Umbral robótico (CV < 0.05)')
        ax.set_title('CV_MONTO_BIN_DIA\n'
                     'Cercano a 0 = todos los montos iguales = comportamiento robótico',
                     fontweight='bold')
        ax.set_xlabel('Coeficiente de variación del monto por BIN×día')
        ax.set_ylabel('Frecuencia')
        ax.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
# Card testing: escala de sospecha BIN10 → BIN11 → BIN12
print('Señales de card testing por longitud de BIN compartido:\n')
for flag, label, descripcion in [
    ('FLAG_BIN10_REPETIDO_DIA', 'BIN10 repetido', 'Sospechoso'),
    ('FLAG_BIN11_REPETIDO_DIA', 'BIN11 repetido', 'Muy sospechoso'),
    ('FLAG_BIN12_REPETIDO_DIA', 'BIN12 repetido', 'Casi seguro generado'),
    ('FLAG_VEN_CONCENTRADA_BIN','Misma fecha vencimiento en ≥3 tarjetas del BIN', 'Generación algorítmica'),
]:
    if flag in df.columns:
        n_flag = int(df[flag].sum())
        if n_flag > 0:
            tasa_f = df.loc[df[flag]==1, 'ES_FRAUDE'].mean() * 100
            lift   = round(tasa_f / tasa_global, 1) if tasa_global > 0 else 0
            print(f'{descripcion:<25} | {label:<45} | {n_flag:>6,} txn | '
                  f'Tasa F: {tasa_f:>5.1f}% | LIFT: {lift:.1f}x')
        else:
            print(f'{descripcion:<25} | {label:<45} | 0 txn')
    else:
        print(f'  ——  {flag} no disponible')

---
## Lista priorizada para el analista

Combina todas las señales sin etiqueta para construir un `SCORE_SOSPECHA`.
Las transacciones N con mayor score son las que deben revisarse primero.

In [ ]:
# Pesos de cada señal (ajustables)
PESOS = {
    'FLAG_MONTO_ROBOTICO_BIN'    : 4,  # monto idéntico = señal más fuerte
    'FLAG_VEN_CONCENTRADA_BIN'   : 4,  # misma fecha vencimiento = generación
    'FLAG_BIN12_REPETIDO_DIA'    : 4,
    'FLAG_BIN11_REPETIDO_DIA'    : 3,
    'FLAG_CLIENTES_BIN_ALTO'     : 3,
    'FLAG_BIN10_REPETIDO_DIA'    : 2,
    'FLAG_RAFAGA_BIN_1H'         : 2,
    'FLAG_RAFAGA_5MIN'           : 2,
    'TIENE_FRAUDE_PREVIO_PERIODO': 2,
    'ES_MADRUGADA'               : 1,
    'FLAG_MONTO_REDONDO'         : 1,
}

pesos_ok = {k: v for k, v in PESOS.items() if k in df.columns}

df['SCORE_SOSPECHA'] = sum(
    df[col].fillna(0).astype(int) * peso
    for col, peso in pesos_ok.items()
)

# Añadir ANOMALY_SCORE si está disponible
if 'ANOMALY_SCORE' in df.columns:
    df['SCORE_SOSPECHA'] += (df['ANOMALY_SCORE'] * 4).round(0).astype(int)

# Top 50 N con mayor score
cols_lista = [c for c in [
    col_ind, col_bin, col_monto, 'SCORE_SOSPECHA',
    'FLAG_MONTO_ROBOTICO_BIN', 'FLAG_BIN11_REPETIDO_DIA',
    'FLAG_VEN_CONCENTRADA_BIN', 'FLAG_CLIENTES_BIN_ALTO',
    'ANOMALY_SCORE', 'SCORE_RIESGO', 'MARCA_TARJETA', 'TIPO_PRODUCTO_TEXTO',
] if c in df.columns]

lista_revision = (
    df[df[col_ind] == 'N']
    .sort_values('SCORE_SOSPECHA', ascending=False)
    .head(50)[cols_lista]
    .reset_index(drop=True)
)
lista_revision.index += 1

print(f'Lista de {len(lista_revision)} transacciones N priorizadas para revisión:')
print(lista_revision.head(10).to_string())

# Exportar
out_path = BASE_DIR / 'output' / f'lista_revision_{COMERCIO_NOMBRE}.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
lista_revision.to_csv(out_path, index=True)
print(f'\n✅ Exportada: {out_path}')

In [ ]:
# Distribución del SCORE_SOSPECHA
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma F vs N
ax = axes[0]
for ind, color, alpha in [('N','#BDD7EE',0.6), ('F','#C00000',0.8)]:
    if ind in df[col_ind].unique():
        _s = df.loc[df[col_ind] == ind, 'SCORE_SOSPECHA']
        ax.hist(_s, bins=20, alpha=alpha, color=color, label=ind, density=True)
ax.set_title('SCORE_SOSPECHA por indicador\n(si F está más a la derecha = funciona)',
             fontweight='bold')
ax.set_xlabel('SCORE_SOSPECHA'); ax.legend()

# N por nivel de score
ax2 = axes[1]
bins_score = [0, 2, 4, 6, 9, 99]
labels_score = ['0-1', '2-3', '4-5', '6-8', '9+']
df['_rango_ss'] = pd.cut(df['SCORE_SOSPECHA'], bins=bins_score, labels=labels_score, right=False)
pivot_ss = df.groupby(['_rango_ss', col_ind]).size().unstack(fill_value=0)
pivot_ss.plot(kind='bar', ax=ax2, color=['#70AD47','#BDD7EE','#FF8C00','#C00000'],
              alpha=0.8, edgecolor='white')
ax2.set_title('N transacciones por rango de SCORE_SOSPECHA', fontweight='bold')
ax2.set_xlabel('Rango SCORE_SOSPECHA'); ax2.set_ylabel('N txn')
ax2.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()
df.drop(columns=['_rango_ss'], inplace=True)

---
## Conclusiones y próximos pasos

### Lo que detectamos sin etiqueta

| Patrón | Variable clave | Señal |
|---|---|---|
| Monto idéntico en tarjetas del mismo BIN | `FLAG_MONTO_ROBOTICO_BIN` | Generación algorítmica |
| Misma fecha de vencimiento en ≥3 tarjetas | `FLAG_VEN_CONCENTRADA_BIN` | Generación algorítmica |
| BIN11/12 repetido entre tarjetas distintas | `FLAG_BIN11/12_REPETIDO_DIA` | Card testing |
| Muchos clientes distintos del mismo BIN en un día | `FLAG_CLIENTES_BIN_ALTO` | Ataque de BIN |
| Transacciones estadísticamente anómalas | `ANOMALY_SCORE > 0.8` | Isolation Forest |

### Escalado a todo ecommerce

Para que este análisis cubra todos los comercios (no solo este):

1. Descargar journals de **todos los comercios ecommerce sin 3DS** del mismo período
2. Consolidarlos en un único parquet (el pipeline ya soporta esto con `MODO_ANALISIS = "MULTI"`)
3. Correr `feature_engineering.py` + `clustering_fraude.py` sobre el consolidado
4. Los patrones de BIN se detectan a través de **todos los comercios**: si el mismo BIN ataca ZARA hoy y Saga mañana, `TRX_BIN_1H` y `CLIENTES_BIN_DIA` lo capturan en el análisis global

### El ciclo de retroalimentación

```
N sospechosa identificada → analista revisa → cierra como F o D
        ↓
Entra a 8850 → actualiza marcaje en 8750
        ↓
Próximo ciclo: Isolation Forest ya conoce ese patrón
        ↓
Detección más temprana en cada iteración
```

**Cada ciclo de revisión mejora el modelo.** No se necesita reentrenar manualmente — el siguiente
journal que bajes ya tendrá las nuevas etiquetas F incorporadas automáticamente.